# Prompt Engineering to Agentic RAG 🤖🧠

## LangChain RAG Agent — Prompt and Context Engineering with AG News

This notebook builds a Retrieval-Augmented Generation agent with LangChain and the public **AG News** dataset. It keeps the retrieval architecture stable while comparing prompt variants and context-optimization strategies.

AG News is historical: treat it as a **news archive**, not a live news service.

## Learning objectives

1. Load and index a public Hugging Face dataset
2. Build a LangChain RAG agent
3. Compare weak and grounded prompts
4. Explain token count and API usage
5. Compare context size, latency, and answer length
7. Apply token-aware truncation, extractive compression, and summarization

## Why context optimization matters

Models process **tokens**, not characters or words. Input usage includes system instructions, conversation history, retrieved documents, tool results, and the user question. Output usage includes the generated answer. More context may increase recall, but it can also increase latency and cost, exceed the context window, and distract the model.

The goal is to send the **smallest context that preserves enough evidence to answer correctly**.

## Key techniques


| Strategy | Extra model call | Strength | Risk |
|---|---:|---|---|
| Full context | No | Maximum detail | High token use and distraction |
| Token truncation | No | Fast and predictable | May omit useful evidence |
| Extractive compression | Embeddings only | Preserves source wording | May lose background |
| LLM summarization | Yes | Strong synthesis and compression | Added latency, cost, information loss |
| Auto policy | Conditional | Balances cost and evidence | Requires evaluation |

## 1. Install dependencies

In [22]:
%pip install -U datasets langchain langchain-openai langchain-community langchain-classic langchain-text-splitters python-dotenv colorama tiktoken sentence-transformers pandas

--- Logging error ---
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 177, in emit
    self.console.print(renderable, overflow="ignore", crop=False, style=style)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1673, in print
    extend(render(renderable, render_options))
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/console.py", line 1305, in render
    for render_output in iter_render:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_internal/utils/logging.py", line 134, in __rich_console__
    for line in lines:
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/pip/_vendor/rich/segment.py", line 249, in split_lines
    for segment in segments:
  File "/Library/

## 2. Environment setup

In [23]:
import os
from getpass import getpass
from colorama import Fore, Style
from dotenv import load_dotenv

load_dotenv(override=True)
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded.{Style.RESET_ALL}")

OpenAI API key loaded.


## 3. Initialize models

In [24]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4.1-mini", temperature=0, timeout=30, max_retries=3)
summary_model = init_chat_model("openai:gpt-4.1-mini", temperature=0, timeout=30, max_retries=3)
print("Models initialized.")

Models initialized.


## 4. Load AG News

In [25]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news", split="train[:500]")
print(dataset)
print(dataset[0])

Dataset({
    features: ['text', 'label'],
    num_rows: 500
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


## 5. Convert data into LangChain documents

In [26]:
from langchain_core.documents import Document

LABELS = {0:"World",1:"Sports",2:"Business",3:"Science and Technology"}
documents=[]
for index,row in enumerate(dataset):
    label=int(row["label"])
    documents.append(Document(page_content=row["text"].strip(), metadata={
        "source":f"ag_news_{index}", "category":LABELS[label],
        "dataset":"AG News", "record_id":index,
    }))
print(f"Documents created: {len(documents)}")
print(documents[0])

Documents created: 500
page_content='Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.' metadata={'source': 'ag_news_0', 'category': 'Business', 'dataset': 'AG News', 'record_id': 0}


## 6. Split, embed, and index

`chunk_overlap` preserves nearby meaning across boundaries, but overlap also duplicates tokens. Tune it instead of increasing it automatically.

In [27]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks=splitter.split_documents(documents)
embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
vector_store=InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)
baseline_retriever=vector_store.as_retriever(search_kwargs={"k":5})
broad_retriever=vector_store.as_retriever(search_kwargs={"k":12})
print(f"Documents: {len(documents)} | Chunks: {len(chunks)}")

Documents: 500 | Chunks: 524


## Token counting and usage

A request roughly consumes:

```text
system prompt + question + retrieved context + history + output = total usage
```

Token counts are useful for setting budgets, comparing strategies, and explaining why larger contexts often cost more and respond more slowly.


In [28]:
import tiktoken
TOKENIZER=tiktoken.encoding_for_model("gpt-4.1-mini")

def count_tokens(text:str)->int:
    return len(TOKENIZER.encode(text))

def document_tokens(doc:Document)->int:
    return count_tokens(doc.page_content)

def inspect_token_usage(docs:list[Document], label="Context"):
    counts=[document_tokens(d) for d in docs]
    stats={
        "label":label, "documents":len(docs), "total_tokens":sum(counts),
        "average_tokens":round(sum(counts)/len(counts),1) if counts else 0,
        "largest_document":max(counts,default=0),
    }
    print(stats)
    return stats

## Formatting Documents

In [29]:
def format_documents(docs:list[Document])->str:
    if not docs:
        return "No relevant articles were retrieved."
    parts=[]
    for i,doc in enumerate(docs,1):
        parts.append(
            f"[Source {i}]\nRecord: {doc.metadata.get('source','unknown')}\n"
            f"Category: {doc.metadata.get('category','unknown')}\nArticle: {doc.page_content}"
        )
    return "\n\n".join(parts)

## Truncation

- Truncation documents based on token count.
- Keeps the highest-ranked documents that fit a fixed token budget. It is fast, deterministic, and adds no LLM call. Its weakness is that useful lower-ranked evidence may be excluded.

In [30]:
def truncate_documents_by_tokens(docs:list[Document], max_tokens=900):
    selected=[]; excluded=[]; used=0
    for doc in docs:
        n=document_tokens(doc)
        if used+n <= max_tokens:
            selected.append(doc); used+=n
        else:
            excluded.append(doc)
    return selected,{
        "budget":max_tokens,"used_tokens":used,
        "selected_documents":len(selected),"excluded_documents":len(excluded),
    }

query="technology companies and business risks"
raw_docs=broad_retriever.invoke(query)
truncated_docs,trunc_report=truncate_documents_by_tokens(raw_docs,900)
inspect_token_usage(raw_docs,"Before truncation")
inspect_token_usage(truncated_docs,"After truncation")
print(trunc_report)

{'label': 'Before truncation', 'documents': 12, 'total_tokens': 606, 'average_tokens': 50.5, 'largest_document': 88}
{'label': 'After truncation', 'documents': 12, 'total_tokens': 606, 'average_tokens': 50.5, 'largest_document': 88}
{'budget': 900, 'used_tokens': 606, 'selected_documents': 12, 'excluded_documents': 0}


## Compression

Extractive compression keeps query-relevant sentences while preserving original wording. It reduces context without a generation call, although sentence-level similarity can remove useful background.

In [31]:
import re
from math import sqrt

def cosine_similarity(a,b):
    numerator=sum(x*y for x,y in zip(a,b))
    na=sqrt(sum(x*x for x in a)); nb=sqrt(sum(y*y for y in b))
    return numerator/(na*nb) if na and nb else 0.0

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+",text) if s.strip()]

def compress_documents(query, docs, threshold=0.20, max_sentences_per_doc=2):
    qv = embeddings.embed_query(query)

    # Split up front so we can batch every sentence across every doc in one call
    doc_sentences = [split_sentences(doc.page_content) for doc in docs]
    all_sentences = [s for sentences in doc_sentences for s in sentences]

    if not all_sentences:
        return []

    # Single batched call instead of one call per document
    all_vectors = embeddings.embed_documents(all_sentences)

    output = []
    idx = 0
    for doc, sentences in zip(docs, doc_sentences):
        n = len(sentences)
        vectors = all_vectors[idx: idx + n]
        idx += n
        if not sentences:
            continue
        ranked = sorted(
            [(cosine_similarity(qv, v), s) for s, v in zip(sentences, vectors)],
            reverse=True,
        )
        kept = [s for score, s in ranked[:max_sentences_per_doc] if score >= threshold]
        if kept:
            output.append(Document(
                page_content=" ".join(kept),
                metadata={**doc.metadata, "compression": "extractive"},
            ))
    return output

compress_docs =compress_documents(query,raw_docs)
compress_docs,_ = truncate_documents_by_tokens(compress_docs,900)
inspect_token_usage(raw_docs,"Raw retrieval")
inspect_token_usage(compress_docs,"Documents compression")
print(format_documents(compress_docs[:3]))

{'label': 'Raw retrieval', 'documents': 12, 'total_tokens': 606, 'average_tokens': 50.5, 'largest_document': 88}
{'label': 'Documents compression', 'documents': 12, 'total_tokens': 511, 'average_tokens': 42.6, 'largest_document': 86}
[Source 1]
Record: ag_news_42
Category: Business
Article: Technology company sues five ex-employees  A Marlborough-based technology company is suing five former employees, including three senior managers, for allegedly conspiring against their employer while working on opening a competing business.

[Source 2]
Record: ag_news_322
Category: Science and Technology
Article: Tech market indeed soft, but HP woes one of a kind HP and Cisco spooked investors with recent comments about sales, but analysts say the overall tech market is still inching along.

[Source 3]
Record: ag_news_310
Category: Science and Technology
Article: If this is a tech rebound, pinch me KnowledgeWharton goes inside the IT conundrum to examine why capital spending--and corporate confiden

## LLM summarization

Summarization can synthesize multiple articles and sharply reduce context, but it adds latency and cost and may blur source boundaries. Apply it conditionally and require the model to preserve source labels and uncertainty.

Retrieved Documents
        │
        ▼
 Count Tokens
   (tiktoken)
        │
        ▼
Token Count
        │
        ▼
Decide whether optimization is needed

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

SUMMARY_PROMPT=ChatPromptTemplate.from_messages([
    ("system",
     "Compress retrieved news evidence for a downstream RAG answer. "
     "Preserve facts, source labels exactly as [Source N], separate events, and limitations. "
     "Do not add outside information. Return no more than {target_words} words."),
    ("human","Question:\n{question}\n\nRetrieved evidence:\n{context}\n\nCompressed evidence:"),
])
summary_chain=SUMMARY_PROMPT | summary_model | StrOutputParser()

def summarize_context(question,docs,target_words=180):
    return summary_chain.invoke({
        "question":question,"context":format_documents(docs),"target_words":target_words
    })

def summarize_if_needed(question,docs,threshold_tokens=1000,target_words=180):
    raw=format_documents(docs); before=count_tokens(raw)
    if before<=threshold_tokens:
        return raw,{"strategy":"full_context","before_tokens":before,"after_tokens":before}
    summary=summarize_context(question,docs,target_words)
    return summary,{"strategy":"llm_summary","before_tokens":before,"after_tokens":count_tokens(summary)}

summarized_context,summary_report=summarize_if_needed(query,raw_docs,700,180)
print(summary_report)
print(summarized_context)

{'strategy': 'llm_summary', 'before_tokens': 837, 'after_tokens': 217}
[Source 1] A Marlborough-based technology company is suing five former employees, including three senior managers, for allegedly conspiring to open a competing business while still employed. [Source 2] HP and Cisco reported sales concerns that spooked investors, but analysts say the overall tech market is still slowly progressing. [Source 3] Corporate confidence and capital spending in IT remain low despite signs of a tech rebound. [Source 6] Lenovo's revenue and profits surged, but it lost market share to multinational competitors. [Source 7] Computer manufacturers and resellers inadequately inform Windows users about internet security risks. [Source 11] Software companies experienced slowed June sales due to factors like high oil prices, steep discounting, and regulatory distractions. [Source 12] Acacia Media Technologies is intensifying efforts to collect streaming video patent fees from colleges and universities

## Context optimizer (truncation, compression and summarization)

In [33]:
# 1) Identify root cause:
#    - Caller passes a string strategy as 3rd positional argument.
#    - optimize_context currently treats 3rd positional argument as max_context_tokens (int).
#    - Comparison int <= str fails.
#
# 2) Make optimize_context strategy-aware and backward-compatible:
#    - Add `strategy` argument.
#    - Keep support for old positional calls where 3rd arg is an int token budget.
#    - Normalize and validate strategy.
#
# 3) Implement explicit strategy branches:
#    - "full": return full formatted context.
#    - "truncate": apply token truncation only.
#    - "extractive": apply extractive compression, then truncation.
#    - "summarize": summarize retrieved docs directly.
#    - "auto": keep current staged policy (compress -> truncate -> optional summarize).
#
# 4) Update the comparison loop:
#    - Pass `strategy=` by keyword to remove positional ambiguity.
#
# 5) (Optional but consistent) Update tool:
#    - Accept a `strategy` parameter and forward it to optimize_context.

from langsmith import traceable

@traceable(name="Optimize Context")
def optimize_context(
    query,
    docs,
    strategy="auto",
    max_context_tokens=900,
    summarize_threshold=700,
):
    """
    Optimize retrieved context with selectable strategies:
    full | truncate | extractive | summarize | auto
    """

    if isinstance(strategy, (int, float)):
        max_context_tokens = int(strategy)
        strategy = "auto"

    strategy = str(strategy).strip().lower()
    valid = {"full", "truncate", "extractive", "summarize", "auto"}
    if strategy not in valid:
        raise ValueError(f"Invalid strategy '{strategy}'. Use one of: {sorted(valid)}")

    before_context = format_documents(docs)
    before_tokens = count_tokens(before_context)

    details = {
        "requested_strategy": strategy,
        "action": "full_context",
        "documents_before": len(docs),
    }

    if strategy == "full":
        context = before_context
        details["action"] = "full_context"

    elif strategy == "truncate":
        selected_docs, truncation_report = truncate_documents_by_tokens(
            docs,
            max_tokens=max_context_tokens,
        )
        context = format_documents(selected_docs)
        details.update(truncation_report)
        details["action"] = "truncate_only"

    elif strategy == "extractive":
        compressed_docs = compress_documents(query, docs)
        details["documents_after_compression"] = len(compressed_docs)

        if not compressed_docs:
            compressed_docs = docs
            details["compression_fallback"] = True

        selected_docs, truncation_report = truncate_documents_by_tokens(
            compressed_docs,
            max_tokens=max_context_tokens,
        )
        context = format_documents(selected_docs)
        details.update(truncation_report)
        details["action"] = "extractive_then_truncate"

    elif strategy == "summarize":
        context = summarize_context(
            query,
            docs,
            target_words=180,
        )
        details["action"] = "summarize_only"

    else:
        if before_tokens <= max_context_tokens:
            context = before_context
            details["action"] = "full_context"
        else:
            compressed_docs = compress_documents(query, docs)
            details["action"] = "extractive_compression"
            details["documents_after_compression"] = len(compressed_docs)

            # Fall back to original docs if compression removes everything
            if not compressed_docs:
                compressed_docs = docs
                details["compression_fallback"] = True

            selected_docs, truncation_report = truncate_documents_by_tokens(
                compressed_docs,
                max_tokens=max_context_tokens,
            )
            context = format_documents(selected_docs)
            details.update(truncation_report)
            details["action"] = "compress_then_truncate"

            if count_tokens(context) > summarize_threshold:
                context = summarize_context(
                    query,
                    selected_docs,
                    target_words=180,
                )
                details["action"] = "compress_truncate_then_summarize"

    after_tokens = count_tokens(context)

    return context, {
        "before_tokens": before_tokens,
        "after_tokens": after_tokens,
        "token_reduction": before_tokens - after_tokens,
        **details,
    }

## 14. Baseline and optimized retrieval tools

In [34]:
from langchain.tools import tool

@tool
def search_news_archive(query:str)->str:
    """Search the historical AG News archive using baseline retrieval."""
    docs=baseline_retriever.invoke(query)
    if not docs: return "STATUS: NO_RESULTS\nNo relevant articles were found."
    context=format_documents(docs)
    return f"STATUS: RESULTS_FOUND\nCONTEXT_STRATEGY: baseline\nCONTEXT_TOKENS: {count_tokens(context)}\n\n{context}"

@tool
def search_optimized_news_archive(query: str, strategy: str = "auto") -> str:
    """Search AG News and optimize the retrieved context automatically."""

    docs = broad_retriever.invoke(query)

    if not docs:
        return (
            "STATUS: NO_RESULTS\n"
            "No relevant articles were found."
        )

    context, report = optimize_context(
        query=query,
        docs=docs,
        strategy=strategy,
    )

    return (
        "STATUS: RESULTS_FOUND\n"
        f"OPTIMIZATION_ACTION: {report.get('action', 'none')}\n"
        f"TOKENS_BEFORE: {report['before_tokens']}\n"
        f"TOKENS_AFTER: {report['after_tokens']}\n"
        f"TOKEN_REDUCTION: {report['token_reduction']}\n\n"
        f"{context}"
    )

## Prompt variations

In [35]:
WEAK_PROMPT="""You are a news assistant. Use the search_optimized_news_archive tool and answer the user's question."""

GROUNDED_PROMPT="""You are a grounded news-archive assistant. The knowledge base is historical, not real-time. Use search_optimized_news_archive with strategy='auto'. Use only retrieved evidence, cite [Source N], and never present archive data as current. If evidence is insufficient, say so. Begin directly, add up to three cited bullets, finish with one limitation, and use no more than 180 words."""

BEGINNER_FRIENDLY_PROMPT="""You are a teacher explaining historical news to beginners. Use search_optimized_news_archive with strategy='auto'. Use only retrieved evidence. Explain difficult terms, use at most three bullets, include one example, cite [Source N], end with 'In simple terms, ...', and write 140 to 180 words."""

EXECUTIVE_PROMPT="""You are preparing a briefing for a business leader. Use search_optimized_news_archive with strategy='extractive'. Return exactly: ## Headline, ## Key Signals, ## Business Relevance, ## Data Limitation. Use only retrieved evidence and no more than 140 words."""

CLARIFYING_PROMPT="""You are a grounded news-archive assistant. If the request is ambiguous, ask one concise clarification question and do not search. Otherwise use search_optimized_news_archive with strategy='auto', cite [Source N], and state when evidence is insufficient."""

ANALYTICAL_PROMPT="""You are an analytical research assistant. Use search_optimized_news_archive with strategy='summarize'. Return exactly: ## Main Finding, ## Evidence, ## Interpretation, ## Limitation. Use only retrieved evidence, neutral language, and no more than 220 words."""

## 16. Create agents

In [36]:
from langchain.agents import create_agent

agents={
    "Weak":create_agent(model=model,tools=[search_optimized_news_archive],system_prompt=WEAK_PROMPT),
    "Grounded":create_agent(model=model,tools=[search_optimized_news_archive],system_prompt=GROUNDED_PROMPT),
    "Beginner":create_agent(model=model,tools=[search_optimized_news_archive],system_prompt=BEGINNER_FRIENDLY_PROMPT),
    "Executive":create_agent(model=model,tools=[search_optimized_news_archive],system_prompt=EXECUTIVE_PROMPT),
    "Clarifying":create_agent(model=model,tools=[search_optimized_news_archive],system_prompt=CLARIFYING_PROMPT),
    "Analytical":create_agent(model=model,tools=[search_optimized_news_archive],system_prompt=ANALYTICAL_PROMPT),
}
print(list(agents))

['Weak', 'Grounded', 'Beginner', 'Executive', 'Clarifying', 'Analytical']


## 17. Invoke agents and measure latency

In [37]:
import time
from langsmith import traceable

@traceable(name="OPTIMIZED RAG Agent")
def ask_optimized_agent(agent,question):
    started=time.perf_counter()
    result=agent.invoke({"messages":[{"role":"user","content":question}]})
    answer=result["messages"][-1].content
    return {
        "answer":answer,"latency_seconds":round(time.perf_counter()-started,3),
        "answer_tokens":count_tokens(answer),
    }

question="What do the articles say about technology companies?"
results=[]
for name in ["Weak","Grounded","Beginner","Executive","Analytical"]:
    result=ask_optimized_agent(agents[name],question)
    results.append({"agent":name,"latency_seconds":result["latency_seconds"],"answer_tokens":result["answer_tokens"]})
    print("\n"+"="*80+f"\n{name.upper()}\n"+"="*80)
    print(result["answer"])


WEAK
The articles about technology companies highlight several key points:

1. The tech market is soft overall, with specific companies like HP and Cisco causing investor concern due to sales issues, but the market is still progressing slowly (Source 1).

2. There is potential for profit in the post-boom tech industry by focusing on operations in slower-growth industries (Source 2).

3. A technology company in Marlborough is suing five former employees for allegedly conspiring to start a competing business (Source 3).

4. Corporate confidence and capital spending in IT remain low despite some signs of a tech rebound (Source 4).

5. Dell is performing well, showing that not all tech companies are struggling (Source 5).

6. A group of technology companies including Texas Instruments, STMicroelectronics, and Broadcom is proposing a new high-speed wireless networking standard (Source 6).

7. The software industry experienced a mysterious sales stall in late June affecting over 20 companie

In [38]:
import pandas as pd
pd.DataFrame(results)

,agent,latency_seconds,answer_tokens
0,Weak,5.711,324
1,Grounded,5.702,225
2,Beginner,8.051,203
3,Executive,4.058,235
4,Analytical,10.944,291


## Compare context strategies directly

In [39]:
context_results=[]
for strategy in ["full","truncate","extractive","summarize","auto"]:
    started = time.perf_counter()
    _,report = optimize_context(question, broad_retriever.invoke(question),strategy)
    context_results.append({
        "strategy":strategy,**report,
        "optimization_latency_seconds":round(time.perf_counter()-started,3),
    })
pd.DataFrame(context_results)

,strategy,before_tokens,after_tokens,token_reduction,requested_strategy,action,documents_before,optimization_latency_seconds,budget,used_tokens,selected_documents,excluded_documents,documents_after_compression
0,full,804,804,0,full,full_context,12,0.292,NaN,NaN,NaN,NaN,NaN
1,truncate,804,804,0,truncate,truncate_only,12,0.191,900.0,577.0,12.0,0.0,NaN
2,extractive,804,757,47,extractive,extractive_then_truncate,12,0.623,900.0,529.0,12.0,0.0,12.0
3,summarize,804,214,590,summarize,summarize_only,12,5.145,NaN,NaN,NaN,NaN,NaN
4,auto,804,804,0,auto,full_context,12,0.254,NaN,NaN,NaN,NaN,NaN


## Ambiguity and insufficient-context tests

In [40]:
print(ask_optimized_agent(agents["Clarifying"],"What happened with Apple?")["answer"])
print("\n---\n")
print(ask_optimized_agent(agents["Grounded"],"What did Apple announce yesterday, and how did its share price react?")["answer"])

Recent news about Apple includes several developments:

- Apple has begun shipping Motion, a high-performance, real-time motion graphics design software integrated with Final Cut Pro HD and DVD Studio Pro 3, priced at $299. [Source 1]
- A Norwegian hacker known for cracking DVD encryption has reportedly broken the locks on Apple's wireless music streaming technology. [Source 2, 4, 9]
- Apple is set to open its second retail store in Japan, located in Osaka, later this month. [Source 3]
- Apple has bundled professional video editing applications, including the new Motion special-effects software. [Source 6]
- The iPod mini has been notably successful in Japan, attributed to its combination of functionality, design, and effective marketing. [Source 7, 12]

If you want information on a specific aspect of Apple, please let me know!

---

There is no relevant information in the retrieved news archive about any announcement Apple made yesterday or how its share price reacted. The available a

## LangChain message trimming

Retrieved documents are not the only source of context growth. Multi-turn history can also exceed the context window. LangChain provides `trim_messages` and approximate token counting for deterministic history control.

In [41]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.messages.utils import trim_messages, count_tokens_approximately

MAX_TOKENS=80

history=[
    HumanMessage(content="Tell me about technology companies."),
    AIMessage(content="Here is a summary of the retrieved articles."),
    HumanMessage(content="Now compare their business risks."),
    AIMessage(content="The articles mention several risks."),
    HumanMessage(content="Which risk appears most often?"),
]
trimmed=trim_messages(history,strategy="last",token_counter=count_tokens_approximately,max_tokens=MAX_TOKENS,start_on="human",end_on=("human","tool"))
print("Messages before:",len(history))
print("Messages after:",len(trimmed))
print("Approximate tokens:",count_tokens_approximately(trimmed))

Messages before: 5
Messages after: 5
Approximate tokens: 69


Example with history of messages exceeding the 80-token threshold

In [42]:
## LangChain message trimming


history=[
    HumanMessage(content="Tell me about technology companies and what risks they currently face in the market."),
    AIMessage(content="Here is a summary of the retrieved articles covering HP, Cisco, Dell, Lenovo, and the Google IPO."),
    HumanMessage(content="Now compare their business risks in more detail, including capital spending and investor confidence."),
    AIMessage(content="The articles mention several risks: soft sales, low capital spending, and cautious investor sentiment."),
    HumanMessage(content="Which risk appears most often across the retrieved articles?"),
    AIMessage(content="Soft sales and cautious investor confidence appear most frequently across the retrieved sources."),
    HumanMessage(content="Can you also mention what Dell reported for its most recent quarter?"),
    AIMessage(content="Dell reported second-quarter earnings that met expectations, with no sales slowdown noted."),
    HumanMessage(content="Which risk appears most often?"),
]

trimmed=trim_messages(
    history,
    strategy="last",
    token_counter=count_tokens_approximately,
    max_tokens=MAX_TOKENS,
    start_on="human",
    end_on=("human","tool"),
)

print("Approximate tokens before:", count_tokens_approximately(history))
print("Messages before:",len(history))
print("Messages after:",len(trimmed))
print("Approximate tokens after:",count_tokens_approximately(trimmed))

Approximate tokens before: 225
Messages before: 9
Messages after: 3
Approximate tokens after: 61


**optimization pipeline**

# Key takeaways

Prompt engineering controls **how the model responds**. Context engineering controls **what evidence the model receives and how many tokens it consumes**. A strong production pipeline usually retrieves broadly, reranks or compresses, enforces a token budget, and summarizes only when necessary.